# YĀTRĀ AI — Notebook 06: Model Comparison & Decision-Threshold Optimization
## Multi-Criteria Model Selection & Production Champion Deployment

This notebook presents the final selection of Yātrā AI's choice prediction engine:
1. **Decision Threshold Sweep** on validation partition ($\tau \in [0.20, 0.60]$)
2. Overcoming positive-class recall suppression caused by class imbalance (2.465 : 1)
3. Selection of **Gradient Boosting Core (GB_core)** at $\mathbf{\tau^* = 0.30}$
4. Evaluation of serialized inference pipeline (`results/models/champion_gb_core.joblib`)


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = os.path.dirname(os.getcwd())
if BASE_DIR not in sys.path:
    sys.path.append(BASE_DIR)

from src.train_models import load_champion_pipeline
print("Inference modules imported.")


### 1. Validation Decision-Threshold Sweep Analysis


In [ ]:
df_thresh = pd.read_csv(os.path.join(BASE_DIR, "results", "tables", "evaluation", "threshold_analysis.csv"))
gb_thresh = df_thresh[df_thresh['model_identifier'] == 'GB_core']
gb_thresh[['threshold', 'val_accuracy', 'val_precision', 'val_recall', 'val_f1', 'val_roc_auc']]


In [ ]:
plt.figure(figsize=(9, 4.5))
plt.plot(gb_thresh['threshold'], gb_thresh['val_f1'], 'o-', label='F1 Score', color='#1f77b4', linewidth=2)
plt.plot(gb_thresh['threshold'], gb_thresh['val_precision'], 's--', label='Precision', color='#2ca02c')
plt.plot(gb_thresh['threshold'], gb_thresh['val_recall'], '^--', label='Recall', color='#d62728')
plt.plot(gb_thresh['threshold'], gb_thresh['val_accuracy'], 'd:', label='Accuracy', color='#7f7f7f')
plt.axvline(0.30, color='black', linestyle='--', label='Tuned Threshold (tau* = 0.30)')
plt.title("Gradient Boosting Core: Validation Metric Trade-offs Across Thresholds", fontsize=11, fontweight='bold')
plt.xlabel("Decision Threshold (tau)")
plt.ylabel("Metric Value")
plt.legend(loc='lower left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


### 2. Final Champion Model Test Performance: Default (0.50) vs. Tuned (0.30)


In [ ]:
df_final = pd.read_csv(os.path.join(BASE_DIR, "results", "tables", "evaluation", "final_results_table.csv"))
df_final


### 3. Feature Importance Analysis
Top 15 features driving the Gradient Boosting decision engine.


In [ ]:
df_fi = pd.read_csv(os.path.join(BASE_DIR, "results", "tables", "models", "feature_importance_summary.csv"))
df_fi.head(15)


### 4. Live Inference with Serialized Champion Pipeline
We load the serialized production inference artifact (`results/models/champion_gb_core.joblib`) and rank candidate alternatives in a real travel session.


In [ ]:
engine = load_champion_pipeline()
print("Champion Pipeline Successfully Loaded!")
print("Model Family:", engine.metadata.get('champion_family'))
print("Operating Decision Threshold:", engine.threshold)

# Load test sample alternatives
df_choice = pd.read_parquet(os.path.join(BASE_DIR, "data", "synthetic", "choice_dataset.parquet"))
sample_session = df_choice[df_choice['session_id'] == df_choice['session_id'].iloc[0]]

ranked = engine.rank_itineraries(sample_session)
print(f"Ranked {len(ranked)} itineraries for Session: {sample_session['session_id'].iloc[0]}")
ranked[['alternative_id', 'mode', 'carrier', 'raw_cost', 'raw_duration', 'predicted_choice_probability', 'predicted_chosen']]
